# Run this cell first

In [ ]:
# this code enables the automated feedback. If you remove this, you won't get any feedback
# so don't delete this cell!
try:
  import AutoFeedback
except (ModuleNotFoundError, ImportError):
  %pip install AutoFeedback
  import AutoFeedback

try:
  from testsrc import test_main
except (ModuleNotFoundError, ImportError):
  %pip install "git+https://github.com/autofeedback-exercises/exercises.git#subdirectory=Hypothesis/Basics"
  from testsrc import test_main

def runtest(tlist):
  import unittest
  from contextlib import redirect_stderr
  from os import devnull
  with redirect_stderr(open(devnull, 'w')):
    suite = unittest.TestSuite()
    for tname in tlist:
      suite.addTest(eval(f"test_main.UnitTests.{tname}"))
    runner = unittest.TextTestRunner()
    try:
      runner.run(suite)
    except AssertionError:
      pass

# Introduction

The exercises in this notebook provide an introduction to the most basic of hypothesis tests. All hypothesis tests work by defining a __null hypothesis__, $H_0$, and an __alternative hypothesis__, $H_1$. The null hypothesis is typically a statement about the distribution that a __test statistic__ is sampled from. The __alternative hypothesis__ is then a statement that says that the __test statistic__ is not a sample from the distribution that was assumed under the __null hypothesis__. 

Having set up the __null and alternative hypothesis__ we then determine a __p-value__ that, if it is small, gives us the evidence that we need to __reject__ the __null hypothesis__ and accept the alternative.  Importantly, however, a large __p-value__ DOES NOT confirm the __null hypothesis__. A large __p-value__ simply tells that that there is not enough evidence to reject the __null hypothesis__ in favour of the __alternative__.

We will explore these further in these exercises by considering a hypothesis test performed on $n$ independent and identically distribution random variables, $X_i$ with known variance $\sigma^2$. The __null hypothesis__ for all our tests will be that the expectation, $\mathbb{E}(X) = \mu$, for the random variables we sampled has a particular value $\mu_0$.  In other words:

$$
H_0: \mu = \mu_0
$$

We will then discuss how to do one and two-tailed hypothesis tests to test this __null hypothesis__ against the following three alternatives.

$$
H_1: \mu < \mu_0 \qquad H_1: \mu \ne \mu_0 \qquad H_1: \mu > \mu_0
$$

The first step in this particular type of hypothesis test is to compute the sample mean, $\overline{X}$, from the sample using:

$$
\overline{X} = \frac{1}{n} \sum_{i=1}^n X_i
$$

We have used python to calculate this quantity in many earlier earlier exercises. However, the first exercise below, nevertheless, begin by reivising how to calculate a sample mean. Before you get on to that, though, you first need to run the cell and the top of this notebook and the cell below to load the various libraries that you will need to complete the exercies. 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats

# Calculating a sample mean

As discussed in the introduction, when we do the particular kind of hypothesis test that we are studying in this notebook we will be given a sample of $n$ independent and identical random variables.  The first step of the hypothesis test then involves computing a sample mean from this data. We will thus begin this exercise by revising how to calculate a sample mean.

The cell below contains the start of a function called `sample_mean` that takes a NumPy array called `sample` in input.  This array contains a set of $n$ independent and identical random variables. Your task is to edit the `sample_mean` function so that it returns a sample mean computed from the data in the NumPy array `sample`.

Remember that you can calculate the sum of a NumPy array using the command `sum` and you can calculate the number of elements in the array by using `len`.  

I have loaded a sample of random variables that I generated into the array called `mysample`. Once you get your code operating correctly the sample mean for this data will be output. 

In [ ]:
def sample_mean( sample ) :
    # Your code for computing the sample mean from the data in sample 
    # goes here
    
    return 1

# You do not need to modify any of the code from here onwards
mysample = np.loadtxt( "sample_data.dat" )
print( sample_mean( mysample ) )

# The central limit theorem revisited

In the last exercise you revised how to write code to evaluate:

$$
\overline{X} = \frac{1}{n}\sum_{i=1}^{n} X_i
$$

where the $X_i$ are the random variables. The data that I provided in the file called `sample_data.dat` were all samples from a normal distribution.  Consequently, the $\overline{X}$ value you calculated is also a normal random variable. To complete the exercise I want you to plot the probability density function (PDF) for $\overline{X}$.

To be clear, you don't need to do any sampling of $\overline{X}$ to plot this PDF and you don't need to calculate a histogram. If you look in your notes from the exercises on random variables you will find that there are analytic expressions for the expectation and variance of $\overline{X}$. I have also told you above that $\overline{X}$ is a normal random variable so you should be able to use the function:

````
yvals = scipy.stats.norm.pdf() 
````

to plot the distribution for $\overline{X}$ here once you have found from your notes what the expectation and variance for this random variable are equal to. The [documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html) for this function may help you with this task.  

To pass the exercise your PDF will need to be evaluated at 100 equally spaced points between -1 and +1.  I have already created an array called `xvals` that contains the $x$ values for these points.  You just need to modify the code that sets the array called `yvals` so that it contains the correct $y$ coordinates for the probability density function.

Notice that the x-axis and y-axis labels need to be "Sample mean" and "Probability density" for you to pass the tests.

In [ ]:
# This generates the 16 standard normal random variabels
samples = np.random.uniform( 0, 1, size=16 )
# Now calculate the sample mean 
xbar = sum(samples) / len(samples)

# Here are the x-values at which I want you to evaluate the probability density
# function for the variable above
xvals = np.linspace( -1, 1, 100 )
# You need to adapt the line below so that the yvals values are the values of the 
# probability density function for the distribution that mu is sampled from.  
# To answer this question you need to remember what the central limit theorem
# tells us about the distribution of a sample mean that is calculated by 
# summing multiple random variables together.
yvals = xvals*xvals

# This will generate the graph for you
plt.plot( xvals, yvals, 'k-')
plt.xlabel("Sample mean")
plt.ylabel("Probability density")
plt.savefig("clt_distribution.png")

# This code is required for the autofeedback- don't delete it!
fighand = plt.gca()

# Test statistic

In the previous exercise we established that the central limit theorem tells us something about the distribution every sample mean computed using:

$$
\overline{X} = \frac{1}{n} \sum_{i=1}^n X_i
$$

If the $X_i$ values in this expression are all identical and independent random variables from a distribution that has a finite expectation and a finite variance then $\overline{X}$ is approximately normally distributed. Furthermore, by looking at your notes, you found out what the expectation, $\mathbb{E}(\overline{X})$, and variance, $\textrm{var}(\overline{X})$, of $\overline{X}$ are equal to.

Given that $\overline{X}$ is a normal random variable we can convert it to a standard normal random variable, $Z$, by employing the standardising transformation:

$$
Z = \frac{\overline{X} - \mathbb{E}(\overline{X}) }{ \sqrt{\textrm{var}(\overline{X})} }
$$

The quantity $Z$ that is returned by this expression is referred to as the __test statistic__ for our hypothesis test.  Your task for this exercise is to complete the function, `teststat`, that I have started in the cell below.  This function should return the test statistic. It takes as input:

* `sample` - the sample of independent and identicaly random variables on which we are performing the hypothesis test
* `mu` - the value for the expectation of the random variables that is assumed under the null hypothesis.
* `sig2` - the variance of the random variables in `sample`. __This is not the same as the $\textrm{var}(\overline{X})$.__

Your function should should return test statistic that you get from applying the transformation described above to the value for `xbar` that I have computed for you using the function that you wrote for computing a sample mean in the first exercise. 

I have loaded a sample of random variables that I generated into the array called `mysample`. Once you get your code operating correctly the test statistic for a hypothesis test in which the null hypothesis is $\mu=0$ will be output.

In [ ]:
def teststat( sample, mu, sig2 ) : 
    xbar = sample_mean( sample )
    # Your code goes here.
    
    return 1

# You do not need to modify any of the code from here onwards
# Notice that I don't really need to load the data here again
# as I read the file sample_data.dat earlier in the notebook
# and set mysample equal to its contents. I am only reading this file here as I have no way of 
# knowing whether you changed the value of the variable mysample
# between the earlier cell where I read it and here.
mysample = np.loadtxt( "sample_data.dat" )
print( teststat( mean, len(mysample), 0, 1 ) )

# The critical region

Hypothesis tests work because, under the assumption that the __null hypothesis__ is true, the __test statistic__ is a sample from a known distribution. For example, in the last exercise the __null hypothesis__ was $\mu=0$. If that is a true statement about the expectation of the random variables in `sample` then the __test_statistic__ should be a standard normal random variable. 

We can use the distribution that is given by the null hypothesis to define a __critical region__. This __critical region__ is setup to ensure that the probability that the __test statistic__ falls within it, if the null hypothesis is true, (a so-called type I error) is given by the __significance level__ of the test (ofen 5%). If the __test statistic__ falls within the __critical region__ the probability the __null hypothesis__ is not true is thus equal to one minus the __significance level__. 

In this exercise, I want you to determine the __critical regions__ for the __test statistic__ that was introduced in the last exercise.  We will define these critical regions for a test on the expectation, $t$, of the __test statistic__. Because $T$ is a sample from a standard normal distribution the expectation, $t$, of $T$ should be zero. We will thus test the __null hypothesis__ $H_0: t=0$ against the three alternatives:

$$
H_1^{(a)}: t<0 \qquad H_1^{(b)}: t \ne 0 \qquad H_1^{(c)}: t>0 
$$

Remember that, if the __null hypothesis__ is true, the __test statistic__, $T$, is standard normal random variable. Consquently:

* If we are testing $H_0$ against $H_1^{(a)}$ we need to find the value $u$ for which $P(T \le u) = 0.05$.
* If we are testing $H_0$ against $H_1^{(b)}$ we need to find the values $u'$ and $l'$ for which $P(T \le u' \vee T \ge l') = 0.05$.
* If we are testing $H_0$ against $H_1^{(c)}$ we need to find the value $l$ for which $P(T \ge l) = 0.05$.

The figure below shows these three critical regions are found if the distribution assumed under the null hypothesis test is a standard normal distribution.  

![hypotestfig1.png](https://raw.githubusercontent.com/autofeedback-exercises/exercises/main/Hypothesis/Basics/hypotestfig1.png)

The top three panels show the PDF for the standard normal distribution, $\phi(x)$, while the bottom three show the cumulative distribution, which is found by evaluating definite integrals from $\phi(x)$ such as the one below:

$$
P(X\le x) = \int_{-\infty}^x \phi(x') \textrm{d}x'
$$

In the top left panel in the figure the shaded region indicates an integral with an upper limit, $x=u$, that ensures the integral is equal to 0.05.  By contrast, in the top right panel the shaded integral is again equal to 0.05 but we are now calculating:

$$
P(X>l) = \int_l^{\infty} \phi(x') \textrm{d}x'
$$

Lastly, in the middle panel we get 0.05 when we add the two integrals that we have shown together and compute the following:

$$
P(X \le u' ) + P(X>l') = \int_{-\infty}^{u'} \phi(x') \textrm{d}x' + \int_{l'}^{\infty} \phi(x') \textrm{d}x' \qquad \textrm{with} \qquad P(X \le u' ) = P(X>l')
$$

The bottom three panels indicate how we read off the values of $u$, $l$, $u'$ and $l'$ using the inverse of the cumulative distribution for the standard normal random variable. In python we can evaluate this inverse function for any $0\le v \le 1$ using:

````
U = scipy.stats.norm.ppf(v)
````

Your task is thus to complete the three functions below:

* `critregH1a` should return the upper bound, $u$, for the critical region for a test of $H_0: t=0$ against $H_1^{(a)}: t<0$ for a hyothesis test with a signifcance level of `siglev`.
* `critregH1a` should return the upper bound, $u'$, for the lower part of the critical region and the lower bound for the upper part, $l'$ of the critical region for a test of $H_0: t=0$ against $H_1^{(a)}: t\ne 0$ for a hyothesis test with a signifcance level of `siglev`.  These two variables must be returned with $u'$ first and $l'$ second.
* `critregH1c` should return the lower bound, $l$, for the critical region for a test of $H_0: t=0$ against $H_1^{(a)}: t>0$ for a hyothesis test with a signifcance level of `siglev`.

In [ ]:
def critregH1a( siglev ) : 
    # You need to add code here
    return 1 

def critregH1b( siglev ) :
    # You need to add code here
    return 1,2

def critregH1c( siglev ) : 
    # You need to add code here
    return 1 

# Calculating the p-value

Determining the extent of the critical region as we did in the previous exercise is a useful exercise for understanding how hypothesis tests operate. It is done relatively rarely in practice, however. The more usual approach is to report the __p-value__ that one obtained from the hypothesis test. This __p-value__ is computed from the value of the __test statistic__. The way it is computed depends on the __null and alternative hypothesis__ that are being tested. In particular:

* The __p-value__ for a test of the __null hypothesis__ $H_0: t=0$ against the __alternative__ $H_1^{(a)}: t <0$ is is equal to $P(X \le T)$.
* The __p-value__ for a test of the __null hypothesis__ $H_0: t=0$ against the __alternative__ $H_1^{(b)}: t \ne 0$ is equal to $P(X \le T) + P(X > T)$.
* The __p-value__ for a test of the __null hypothesis__ $H_0: t=0$ against the __alternative__ $H_1^{(c)}: t >0$ is equal to $P(X > T)$.

For the type of test we are doing, $X$ in all these statements is a standard normal random variable because, if we assume the __null hypothesis__ is true, $T$ is a standard normal random variable.  

The figure below illustrates how these __p-values__ are computed from the __test statistic__, $T$, using the the probability density and cumulative distribution for this type of random variable:

![hypotestfig2.png](https://raw.githubusercontent.com/autofeedback-exercises/exercises/main/Hypothesis/Basics/hypotestfig2.png)

You can see that if we are evaluating $H_0: t=0$ against $H_1^{(a)}: t <0$ then we calculate the __p-value__ from the __test statistic__, $T$, using:

$$
p = \int_{-\infty}^T \phi(x) \textrm{d}x
$$

If we are evaluating $H_0: t=0$ against $H_1^{(c)}: t >0$ then we calculate the __p-value__ from the __test statistic__, $T$, using:

$$
p = \int_{T}^\infty \phi(x) \textrm{d}x
$$

Lastly, if we evaluating $H_0: t=0$ against $H_1^{(b)}: t \ne 0$ then we calculate the __p-value__ from the __test statistic__, $T$, using:

$$
p = \int_{-\infty}^{|T|} \phi(x) \textrm{d}x + \int_{|T|}^\infty \phi(x) \textrm{d}x
$$

where $|T|$ is the absoluate value of $T$ and $\phi(x)$ is the probility density for a standard normal random variable.

Calculating the __p-value__ in these ways allows us to intepret it as the probability of a __false positive__ result.  In other words:

* If we are testing the __null hypothesis__ $H_0: t=0$ against the __alternative__ $H_1^{(a)}: t <0$ the __p-value__ gives us $P(T'\le T|H_0=1)$, where $T'$ is second sample of the test statistic.   
* If we are testing the __null hypothesis__ $H_0: t=0$ against the __alternative__ $H_1^{(b)}: t \ne 0$ the __p-value__ gives us $P(T'\le T|H_0=1)+P(T'>T|H_0=1)$, where $T'$ is second sample of the test statistic.
* If we are testing the __null hypothesis__ $H_0: t=0$ against the __alternative__ $H_1^{(c)}: t >0$ the __p-value__ gives us $P(T'> T|H_0=1)$, where $T'$ is second sample of the test statistic.

Notice that the condition $H_0=1$ in these expressions tells us that we are assuming that the __null hypothesis__ is true when interpretting these probabilities. Consequently, we interpret 1 minus the __p-value__ as a probablity that the __alternative hypothesis__ is true in frequentist statistics despite the fact that:

$$
1 - P(T'\le T|H_0=1) = P(T'>T|H_0=1)
$$

## The exercise

You now have all the information you need to caclualte the __p-values__ for these three type of hypothesis test.  Your task is thus to complete the three functions below. Each of these functions takes the following three variables in input:

* `sample` - the sample of independent and identicaly random variables on which we are performing the hypothesis test
* `mu0` - the value for the expectation of the random variables that is assumed under the null hypothesis.
* `sig2` - the variance of the random variables in `sample`. __This is not the same as the $\textrm{var}(\overline{X})$.__

You will also see that I have called the function for calculating the test statistic, `teststat`, that you wrote in an earlier exercise within each of them.  Your task is thus to convert the value of the __test statistic__, `t`, that is returned by this function to a __p-value__ for each type of test. For the three functions the __null, $H_0$ and alternative hypothesis, $H_1$,__ are:

* `pval_lower` - $H_0: \mu=\mu_0$ against $H_1: \mu<\mu_0$
* `pval_not` - $H_0: \mu=\mu_0$ against $H_1: \mu\ne \mu_0$
* `pval_higher`- - $H_0: \mu=\mu_0$ against $H_1: \mu>\mu_0$

Where $\mu_0$ is the value of the input variable `mu0`.  

I have included code at the end of the cell below to perform these hypothesis tests with $\mu_0=0$ on the data set that we have analysed in earlier exercises.

In [ ]:
def pval_lower( sample, mu0, sig2 ) : 
    t = teststat( sample, mu0, sig2 )
    # Your code goes here


def pval_not( sample, mu0, sig2 ) : 
    t = teststat( sample, mu0, sig2 )
    # Your code goes here


def pval_higher( sample, mu0, sig2 ) : 
    t = teststat( sample, mu0, sig2 )
    # Your code goes here


# You do not need to modify any of the code from here onwards
# Notice that I don't really need to load the data here again
# as I read the file sample_data.dat earlier in the notebook
# and set mysample equal to its contents. I am only reading this file here as I have no way of 
# knowing whether you changed the value of the variable mysample
# between the earlier cell where I read it and here.
mysample = np.loadtxt( "sample_data.dat" )
print( "The p-value for a test of H_0: mu=0 against H_1 mu<0", pval_lower( mysample, 0, 1) )
print( "The p-value for a test of H_0: mu=0 against H_1 mu \ne 0", pval_not( mysample, 0, 1) )
print( "The p-value for a test of H_0: mu=0 against H_1 mu>0", pval_higher( mysample, 0, 1) )

# Sampling p-values and false negatives

You now know enough to sample the distribution of __p-value__ that are returned when you perform a hypothesis test as I did in the example report.  As illustrated in the example report, it is useful to do sampling like this as it allows you to compute the probablity of __a false negative__ result.  

For the following exercise, you will need to complete the function, `false_neg_rate` that I have started below.  This function should return an estimate the probability of a __false negative__ by generating `ntest` samples that each contain `n` normal random variables from a distribution with an expectation of `mu1` and variance `sig2`. You will compute a __test statistic__ and a __p-value__ for each of  these `ntest` samples of the __null and alternative hypothesis__ tests below:

$$
H_0: \mu = \mu_0 \qquad H_1: \mu < \mu_0
$$

where $\mu_0$ is equal to the value of the variable `mu0` that was input to the function. Notice, that the values input for `mu1` is less than that input for `mu0` so the alternative hypothesis is true for all the samples that you will generate.

Your function should return the fraction of times the __p-value__ in your `ntest` hypothesis tests was greater than `significance` - the __significance level__ for the hypothesis test.  I would also strongly recommend using one of the functions that you wrote in the last exercises for calculating the `ntest` __p-values__.  

I have written code at the end of the cell below to estimate the false negative rate for 200 tests of the null hypothesis $H_0: \mu=0.2$ against the alternative $H_1: \mu<0.2$, which computes test statistics from samples of 
100 standard normal random variables.

_Notice, that the final value that your code return is an estimator for the $p$ parameter of a Bernoulli random variable.  If you were reporting the false positive rate in a report you would need to compute a confidence interval on the estimate of this quantity that you obtained by sampling. I am not asking you to do this here but I have shown you how this can be done in other exercises and would expect to see the confidence limit in any report you wrote using these ideas._    


In [ ]:
def false_neg_rate( ntest, n, mu0, mu1, sig2, significance  ):
    # Your code goes here
    return 1


# Once you finish the function above this code should print out the number of
# times a hypothesis test of H_0: mu=0.2 against H_1: mu<0.2 returns a p-value greater than 0.05.
# In each of these hypothesis tests the test statistic is computed from 100 standard normal 
# random variables. 
print( false_neg_rate( 200, 100, 0.2, 0, 1, 0.05) )